In [2]:

# Section 1: Load E5 Training Data and Group-Aware Split

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import joblib

PROJECT_ROOT = Path.home() / "Desktop" / "Erdos-Router"

DATA = (
    PROJECT_ROOT / "1_data_preparation"
    / "data" / "processed" / "char_tfidf_50k"
)

E5_DATA = (
    PROJECT_ROOT / "1_data_preparation"
    / "data" / "processed" / "e5_small"
)

# Load original training partition only
X_tfidf_train = sparse.load_npz(
    DATA / "X_train.npz"
).tocsr()

cost_train = pd.read_pickle(DATA / "train_costs.pkl")
train_questions = pd.read_pickle(DATA / "train_questions.pkl")
preprocessing = joblib.load(DATA / "preprocessing.joblib")

MODELS = list(cost_train.columns)

# Load saved E5 embeddings
E5_train = np.load(
    E5_DATA / "E5_train.npy",
    allow_pickle=False
)

# Extract scaled character-length feature
X_train_length = X_tfidf_train[:, -1].toarray()

# Assemble E5 + length features
X_train_e5 = np.hstack([
    E5_train,
    X_train_length
])

# Load exact split used for Gamma TF-IDF
split_path = (
    PROJECT_ROOT / "saved_models"
    / "training_splits"
    / "routerbench_grouped_subtrain_subval_seed42.npz"
)

with np.load(split_path) as split:
    subtrain_idx = split["subtrain_idx"]
    subval_idx = split["subval_idx"]

X_subtrain_e5 = X_train_e5[subtrain_idx]
X_subval_e5 = X_train_e5[subval_idx]

cost_subtrain = cost_train.iloc[subtrain_idx].copy()
cost_subval = cost_train.iloc[subval_idx].copy()

# Verification
assert X_train_e5.shape == (21900, 385)
assert X_subtrain_e5.shape == (17520, 385)
assert X_subval_e5.shape == (4380, 385)
assert len(MODELS) == 11
assert np.isfinite(X_train_e5).all()
assert (cost_train.to_numpy() > 0).all()

groups = train_questions["question_group"].to_numpy()
assert set(groups[subtrain_idx]).isdisjoint(
    set(groups[subval_idx])
)

print("E5 + length training:", X_train_e5.shape)
print("Sub-training:", X_subtrain_e5.shape)
print("Sub-validation:", X_subval_e5.shape)
print("LLMs:", len(MODELS))
print("\nE5 training data verified.")


E5 + length training: (21900, 385)
Sub-training: (17520, 385)
Sub-validation: (4380, 385)
LLMs: 11

E5 training data verified.


In [3]:

# Section 2.1: Gamma Deviance Loss

import numpy as np

def gamma_deviance(y_true, y_pred):
    """Compute mean Gamma deviance for positive costs."""

    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    if y_true.shape != y_pred.shape:
        raise ValueError("Observed and predicted shapes must match.")

    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError("Costs and predictions must be finite.")

    if (y_true <= 0).any() or (y_pred <= 0).any():
        raise ValueError("Costs and predictions must be positive.")

    log_ratio = np.log(y_true) - np.log(y_pred)

    return float(
        2.0 * np.mean(np.expm1(log_ratio) - log_ratio)
    )


# Simple verification
actual = np.array([1.0, 2.0, 3.0])

perfect = np.array([1.0, 2.0, 3.0])
imperfect = np.array([1.2, 1.8, 2.5])

print("Perfect predictions:", gamma_deviance(actual, perfect))
print("Imperfect predictions:", gamma_deviance(actual, imperfect))


Perfect predictions: 0.0
Imperfect predictions: 0.0260559525244121


In [4]:

# Section 2.2: Regularized Gamma Objective

def gamma_objective(y_true, mu_pred, coefficients, lam):
    """Half mean Gamma deviance plus L2 regularization."""

    if lam < 0:
        raise ValueError("lambda must be nonnegative.")

    coefficients = np.asarray(coefficients, dtype=np.float64)

    data_loss = 0.5 * gamma_deviance(y_true, mu_pred)
    penalty = 0.5 * lam * np.dot(coefficients, coefficients)

    return float(data_loss + penalty)


actual = np.array([1.0, 2.0, 3.0])
predicted = np.array([1.2, 1.8, 2.5])

beta = np.array([0.5, -0.3, 0.2])
lam = 0.1

print("Regularized objective:",
      gamma_objective(actual, predicted, beta, lam))


Regularized objective: 0.032027976262206055


In [5]:

# Section 2.3: Gamma Objective and Analytical Gradient

def gamma_loss_and_gradient(params, X, y, lam):
    """
    Compute regularized Gamma objective and gradient.

    params[0]  : intercept
    params[1:] : coefficients
    """

    params = np.asarray(params, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    if lam < 0:
        raise ValueError("lambda must be nonnegative.")

    if not np.isfinite(y).all() or (y <= 0).any():
        raise ValueError("Observed costs must be finite and positive.")

    intercept = params[0]
    beta = params[1:]

    n = X.shape[0]

    eta = intercept + X @ beta
    log_ratio = np.log(y) - eta

    # Stable formulation for typical finite log-ratios
    ratio_minus_one = np.expm1(log_ratio)

    data_loss = np.mean(ratio_minus_one - log_ratio)
    penalty = 0.5 * lam * np.dot(beta, beta)

    loss = float(data_loss + penalty)

    residual = -ratio_minus_one

    grad_intercept = np.mean(residual)

    grad_beta = (
        np.asarray(X.T @ residual).ravel() / n
        + lam * beta
    )

    gradient = np.concatenate([
        [grad_intercept],
        grad_beta
    ])

    if not np.isfinite(loss) or not np.isfinite(gradient).all():
        raise FloatingPointError("Non-finite Gamma objective or gradient.")

    return loss, gradient


In [6]:

# Section 2.4: Numerical Gradient Check

from scipy.optimize import check_grad

X_small = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

y_small = np.array([1.0, 2.0, 3.0])

params_small = np.array([0.2, 0.1, -0.1])

lam = 0.1

def objective_only(params):
    return gamma_loss_and_gradient(
        params, X_small, y_small, lam
    )[0]

def gradient_only(params):
    return gamma_loss_and_gradient(
        params, X_small, y_small, lam
    )[1]

error = check_grad(
    objective_only,
    gradient_only,
    params_small
)

print("Gradient-check error:", error)


Gradient-check error: 8.591180626857736e-09


In [7]:

# Section 3.1: Test L-BFGS Optimization — Gamma E5

import numpy as np
from scipy.optimize import minimize

model_name = "meta/llama-2-70b-chat"

# Use only 1,000 sub-training observations
X_fit = X_subtrain_e5[:1000]
y_fit = cost_subtrain[model_name].to_numpy()[:1000]

# Initialize the intercept at log mean cost
initial_params = np.zeros(X_fit.shape[1] + 1)
initial_params[0] = np.log(y_fit.mean())

# Fixed regularization strength
lam = 0.1

initial_loss, _ = gamma_loss_and_gradient(
    initial_params, X_fit, y_fit, lam
)

result = minimize(
    fun=gamma_loss_and_gradient,
    x0=initial_params,
    args=(X_fit, y_fit, lam),
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 50,
        "ftol": 1e-9
    }
)

print("Initial objective:", initial_loss)
print("Final objective:", result.fun)
print("Iterations:", result.nit)
print("Converged:", result.success)
print("Message:", result.message)

assert np.isfinite(result.fun)
assert result.fun <= initial_loss + 1e-10


Initial objective: 0.285179166429334
Final objective: 0.10713930384963037
Iterations: 15
Converged: True
Message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH


In [8]:

# Section 3.2: Gamma E5 Training with Early Stopping

from scipy.optimize import minimize
import numpy as np
import pandas as pd

GAMMA_LAMBDA = 0.1


def train_gamma_early_stopping(
    X_fit,
    y_fit,
    X_holdout,
    y_holdout,
    lam=GAMMA_LAMBDA,
    max_iter=200,
    patience=5,
    min_delta=1e-6
):
    """
    Train Gamma regression using L-BFGS-B.

    Fit partition: internal sub-training only.
    Holdout partition: internal sub-validation only.

    Returns the checkpoint with the lowest observed
    sub-validation Gamma deviance.
    """

    y_fit = np.asarray(y_fit, dtype=np.float64)
    y_holdout = np.asarray(y_holdout, dtype=np.float64)

    if X_fit.shape[1] != X_holdout.shape[1]:
        raise ValueError("Feature dimensions must agree.")

    initial_params = np.zeros(X_fit.shape[1] + 1)
    initial_params[0] = np.log(y_fit.mean())

    history = []

    best_params = initial_params.copy()
    best_loss = np.inf
    best_iteration = 0

    no_improvement = 0
    iteration = 0

    def record_checkpoint(params, iteration):
        nonlocal best_params, best_loss
        nonlocal best_iteration, no_improvement

        eta_fit = params[0] + X_fit @ params[1:]
        eta_holdout = params[0] + X_holdout @ params[1:]

        fit_deviance = gamma_deviance(
            y_fit, np.exp(eta_fit)
        )

        holdout_deviance = gamma_deviance(
            y_holdout, np.exp(eta_holdout)
        )

        history.append({
            "iteration": iteration,
            "subtrain_deviance": fit_deviance,
            "subval_deviance": holdout_deviance
        })

        # Retain the lowest observed sub-validation loss
        if holdout_deviance < best_loss:
            best_params = params.copy()
            best_loss = holdout_deviance
            best_iteration = iteration

        return holdout_deviance

    # Record initialization (iteration 0)
    previous_best = np.inf
    record_checkpoint(initial_params, 0)
    previous_best = best_loss

    class EarlyStop(Exception):
        pass

    def callback(params):
        nonlocal iteration, no_improvement
        nonlocal previous_best

        iteration += 1

        record_checkpoint(params, iteration)

        if best_loss < previous_best - min_delta:
            no_improvement = 0
            previous_best = best_loss
        else:
            no_improvement += 1

        if no_improvement >= patience:
            raise EarlyStop()

    stopped_early = False

    try:
        result = minimize(
            fun=gamma_loss_and_gradient,
            x0=initial_params,
            args=(X_fit, y_fit, lam),
            jac=True,
            method="L-BFGS-B",
            callback=callback,
            options={
                "maxiter": max_iter,
                "ftol": 1e-12,
                "gtol": 1e-8
            }
        )
    except EarlyStop:
        stopped_early = True
        result = None

    history_df = pd.DataFrame(history)

    return {
        "params": best_params,
        "history": history_df,
        "best_iteration": best_iteration,
        "best_subval_deviance": best_loss,
        "stopped_early": stopped_early,
        "optimizer_result": result
    }


print("Gamma E5 early-stopping function defined.")


Gamma E5 early-stopping function defined.


In [9]:

# Section 4.1: Train All 11 Gamma E5 Models

gamma_e5_final_models = {}
gamma_e5_training_histories = {}
gamma_e5_training_records = []

for model_name in MODELS:

    print(f"Training Gamma E5: {model_name}")

    result = train_gamma_early_stopping(
        X_fit=X_subtrain_e5,
        y_fit=cost_subtrain[model_name].to_numpy(),
        X_holdout=X_subval_e5,
        y_holdout=cost_subval[model_name].to_numpy(),
        lam=GAMMA_LAMBDA,
        max_iter=200,
        patience=5,
        min_delta=1e-6
    )

    # Save the best checkpoint in memory
    gamma_e5_final_models[model_name] = result["params"].copy()

    # Save the training history in memory
    gamma_e5_training_histories[model_name] = result["history"].copy()

    # Record training results
    gamma_e5_training_records.append({
        "model": model_name,
        "lambda": GAMMA_LAMBDA,
        "best_iteration": result["best_iteration"],
        "best_subval_deviance": result["best_subval_deviance"],
        "iterations_recorded": len(result["history"]) - 1,
        "stopped_early": result["stopped_early"]
    })

gamma_e5_training_summary = pd.DataFrame(
    gamma_e5_training_records
)

print("\nAll 11 Gamma E5 models trained.")
display(gamma_e5_training_summary)


Training Gamma E5: WizardLM/WizardLM-13B-V1.2
Training Gamma E5: claude-instant-v1
Training Gamma E5: claude-v1
Training Gamma E5: claude-v2
Training Gamma E5: gpt-3.5-turbo-1106
Training Gamma E5: gpt-4-1106-preview
Training Gamma E5: meta/code-llama-instruct-34b-chat
Training Gamma E5: meta/llama-2-70b-chat
Training Gamma E5: mistralai/mistral-7b-chat
Training Gamma E5: mistralai/mixtral-8x7b-chat
Training Gamma E5: zero-one-ai/Yi-34B-Chat

All 11 Gamma E5 models trained.


,model,lambda,best_iteration,best_subval_deviance,iterations_recorded,stopped_early
0,WizardLM/WizardLM-13B-V1.2,0.1,8,0.149683,13,True
1,claude-instant-v1,0.1,8,0.287060,13,True
2,claude-v1,0.1,8,0.265727,13,True
3,claude-v2,0.1,8,0.333969,13,True
4,gpt-3.5-turbo-1106,0.1,8,0.162770,13,True
5,gpt-4-1106-preview,0.1,8,0.448028,13,True
6,meta/code-llama-instruct-34b-chat,0.1,9,0.099903,14,True
7,meta/llama-2-70b-chat,0.1,7,0.136604,12,True
8,mistralai/mistral-7b-chat,0.1,8,0.122777,13,True
9,mistralai/mixtral-8x7b-chat,0.1,7,0.125019,12,True


In [10]:

# Section 5.1: Verify All Gamma E5 Checkpoints

import numpy as np
import pandas as pd

verification_records = []

for model_name in MODELS:

    params = gamma_e5_final_models[model_name]
    history = gamma_e5_training_histories[model_name]

    # 385 features plus one intercept
    assert params.shape == (386,)
    assert np.isfinite(params).all()

    # Predict on INTERNAL sub-validation only
    eta = params[0] + X_subval_e5 @ params[1:]
    predicted_cost = np.exp(eta)

    assert np.isfinite(predicted_cost).all()
    assert (predicted_cost > 0).all()

    y_subval = cost_subval[model_name].to_numpy()

    # Independently recompute Gamma deviance
    verified_deviance = gamma_deviance(
        y_subval,
        predicted_cost
    )

    # Compare with the recorded best checkpoint
    recorded_deviance = history["subval_deviance"].min()

    np.testing.assert_allclose(
        verified_deviance,
        recorded_deviance,
        rtol=1e-8,
        atol=1e-10
    )

    verification_records.append({
        "model": model_name,
        "parameter_count": len(params),
        "verified_subval_deviance": verified_deviance,
        "verified": True
    })

gamma_e5_verification = pd.DataFrame(verification_records)

display(gamma_e5_verification)

print("\nAll 11 Gamma E5 checkpoints verified successfully.")


,model,parameter_count,verified_subval_deviance,verified
0,WizardLM/WizardLM-13B-V1.2,386,0.149683,True
1,claude-instant-v1,386,0.287060,True
2,claude-v1,386,0.265727,True
3,claude-v2,386,0.333969,True
4,gpt-3.5-turbo-1106,386,0.162770,True
5,gpt-4-1106-preview,386,0.448028,True
6,meta/code-llama-instruct-34b-chat,386,0.099903,True
7,meta/llama-2-70b-chat,386,0.136604,True
8,mistralai/mistral-7b-chat,386,0.122777,True
9,mistralai/mixtral-8x7b-chat,386,0.125019,True



All 11 Gamma E5 checkpoints verified successfully.


In [11]:

# Section 6.1: Save Final Gamma E5 Models and Training Records

from pathlib import Path
import joblib
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "saved_models" / "gamma_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Combine all 11 training histories
gamma_e5_training_history = pd.concat(
    [
        history.assign(model=model_name)
        for model_name, history in gamma_e5_training_histories.items()
    ],
    ignore_index=True
)

# Package trained models with reproducibility metadata
gamma_e5_bundle = {
    "model_type": "gamma_log_link_l2",
    "feature_type": "e5_small_384_plus_scaled_char_length",
    "model_names": list(MODELS),
    "models": gamma_e5_final_models,
    "lambda": GAMMA_LAMBDA,
    "training_summary": gamma_e5_training_summary.copy(),
    "feature_count": 385,
    "parameter_count": 386,
    "embedding_model": "intfloat/multilingual-e5-small",
    "embedding_file": "1_data_preparation/data/processed/e5_small/E5_train.npy",
    "length_preprocessing": preprocessing["length_scaler"],
    "feature_order": ["e5_embedding", "scaled_char_length"],
    "early_stopping": {
        "patience": 5,
        "min_delta": 1e-6,
        "max_iter": 200,
        "monitor": "subvalidation_gamma_deviance"
    },
    "training_split_file": (
        "saved_models/training_splits/"
        "routerbench_grouped_subtrain_subval_seed42.npz"
    ),
    "prediction_target": "expected_inference_cost",
    "prediction_link": "exponential",
    "calibration_status": "not_calibrated",
    "data_usage": "original_training_partition_only"
}

# Output paths
model_path = OUTPUT_DIR / "gamma_e5_length_early_stopped.joblib"
history_path = OUTPUT_DIR / "gamma_e5_length_training_history.csv"
summary_path = OUTPUT_DIR / "gamma_e5_length_training_summary.csv"

# Save the files
joblib.dump(gamma_e5_bundle, model_path)

gamma_e5_training_history.to_csv(
    history_path, index=False
)

gamma_e5_training_summary.to_csv(
    summary_path, index=False
)

print("Saved Gamma E5 artifacts:\n")

for path in [model_path, history_path, summary_path]:
    print(path.name, "-", path.stat().st_size, "bytes")

print("\nAll 11 trained Gamma E5 models saved successfully.")


Saved Gamma E5 artifacts:

gamma_e5_length_early_stopped.joblib - 38461 bytes
gamma_e5_length_training_history.csv - 9662 bytes
gamma_e5_length_training_summary.csv - 688 bytes

All 11 trained Gamma E5 models saved successfully.


In [13]:

# Section 7.1: Reload and Verify Saved Gamma E5 Models

import joblib
import numpy as np

# Load saved model bundle
loaded_bundle = joblib.load(
    OUTPUT_DIR / "gamma_e5_length_early_stopped.joblib"
)

# Verify metadata
assert loaded_bundle["model_type"] == "gamma_log_link_l2"
assert loaded_bundle["feature_count"] == 385
assert loaded_bundle["parameter_count"] == 386
assert loaded_bundle["calibration_status"] == "not_calibrated"

assert loaded_bundle["model_names"] == list(MODELS)
assert set(loaded_bundle["models"]) == set(MODELS)

# Verify every saved parameter vector
for model_name in MODELS:

    original_params = gamma_e5_final_models[model_name]
    loaded_params = loaded_bundle["models"][model_name]

    np.testing.assert_array_equal(
        original_params,
        loaded_params
    )

    # Compare predictions using INTERNAL sub-validation only
    original_predictions = np.exp(
        original_params[0]
        + X_subval_e5 @ original_params[1:]
    )

    loaded_predictions = np.exp(
        loaded_params[0]
        + X_subval_e5 @ loaded_params[1:]
    )

    np.testing.assert_array_equal(
        original_predictions,
        loaded_predictions
    )

print("Saved Gamma E5 bundle reloaded successfully.")
print("Models verified:", len(MODELS))
print("Parameters identical: Yes")
print("Predictions identical: Yes")
print("Calibration status:", loaded_bundle["calibration_status"])


Saved Gamma E5 bundle reloaded successfully.
Models verified: 11
Parameters identical: Yes
Predictions identical: Yes
Calibration status: not_calibrated
